# 과제 1 — 시연 데이터 수집 및 Hugging Face Hub 업로드

이 노트북은 SO-101 Leader/Follower 로봇으로 수집한 시연 데이터셋을 Hugging Face Hub에서 확인하고, 50개 에피소드의 메타데이터와 검수 결과를 정리하기 위한 제출용 노트북입니다.

> **현재 진행 상태:** 지금까지 확인된 성공 업로드는 `rimur01/so101-lab5`이며, 카메라 0번으로 1회(20초) 기록한 데이터입니다. 과제 조건인 **공개 데이터셋 50 에피소드와 두 카메라 영상**을 충족하려면, 아래 녹화 명령으로 최종 데이터셋을 완성해 Hub에 업로드한 후 이 노트북의 셀을 다시 실행해야 합니다. 검수표도 실제 재생 검수 후에만 `O` 또는 `X`로 채우세요. 완료하지 않은 조건을 통과로 표시하지 않도록 구성했습니다.

GitHub에는 이 노트북만 올리고, 동영상·Parquet 데이터 파일은 GitHub 저장소에 넣지 마세요.


## 0. 제출 정보

학번과 이름은 본인 정보로 입력하세요. `DATASET`은 최종 제출할 공개 Hub 저장소 하나만 지정합니다.


In [ ]:
# [작성] 본인 정보로 수정
STUDENT_ID = "학번 입력"
NAME = "이름 입력"
HF_USER = "rimur01"

# 50개 에피소드와 두 카메라를 포함해 최종 업로드할 저장소
# 실제 업로드한 repo_id가 다르면 실제 이름으로 수정하세요.
DATASET = f"{HF_USER}/so101-lab5-2cam"
TASK = "Pick up an object and place it in a target area"  # 녹화 시 --dataset.single_task에 넣은 문장과 정확히 같아야 함

print("학번/이름      :", STUDENT_ID, NAME)
print("데이터셋 repo_id:", DATASET)
print("데이터셋 링크  :", f"https://huggingface.co/datasets/{DATASET}")
print("태스크 설명    :", TASK)


## Hub 접근 확인

다음 셀은 선택한 저장소의 파일 목록과 공개 여부를 확인합니다. 최종 데이터셋을 Hub에 업로드한 뒤 실행하세요.


In [ ]:
import json
import requests
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
repo_files = []
info_api = None
is_public = False
try:
    # 익명 HTTP 요청으로 공개 여부를 확인합니다.
    response = requests.get(f"https://huggingface.co/api/datasets/{DATASET}", timeout=20)
    if response.status_code == 200:
        info_api = response.json()
        is_public = not bool(info_api.get("private", False))
    repo_files = api.list_repo_files(DATASET, repo_type="dataset")
    if is_public:
        print(f"OK  {DATASET}  (파일 {len(repo_files)}개, public=True)")
    else:
        print(f"FAIL  {DATASET}: 비공개 저장소이거나 공개 상태를 확인하지 못함")
except Exception as e:
    print(f"FAIL  {DATASET} -> {type(e).__name__}: repo_id와 Hub 업로드 상태를 확인하세요.")


## 1. 태스크 및 수집 환경

### 1-1. 태스크 정의

- **태스크 설명:** 책상 위의 물체를 집어 지정된 목표 위치에 놓는다.
- **성공 조건:** 물체를 그리퍼로 집어 목표 영역에 놓고, 동작이 끝난 뒤 로봇을 안전한 대기 자세로 이동한다.
- **시작 조건:** 로봇과 그리퍼가 안전한 시작 자세에 있고, 물체가 카메라 화면 안의 그리퍼 도달 범위에 놓여 있다.
- **태스크 문자열:** `Pick up an object and place it in a target area` (녹화 명령어와 데이터셋 메타데이터에서 동일해야 함)

### 1-2. 수집 환경

- 로봇: SO-101 Leader / Follower
- Windows 포트: Leader `COM7`, Follower `COM6`
- 카메라: OpenCV 인덱스 `0` 및 `2`
- 성공한 동시 카메라 프레임 읽기 테스트: Windows DSHOW + MJPG 설정, 640×480, 15 FPS
- Hugging Face 계정: `rimur01`
- 최종 저장소: `rimur01/so101-lab5-2cam`

> 두 카메라의 동시 프레임 읽기 테스트는 성공했지만, 두 카메라를 사용한 LeRobot 녹화와 Hub 업로드는 별도로 확인해야 합니다. 실제 최종 녹화 설정이 다르면 이 설명을 실제 값으로 고치세요.

### 1-3. 녹화 명령어

아래 명령어는 Windows CMD에서 사용하도록 작성했습니다. `num_episodes=50`, `push_to_hub=true`, `private=false`가 중요합니다. 각 에피소드가 성공했는지 확인한 후 다음 에피소드로 넘어가세요.


In [ ]:
# [실제 실행한 명령어] 최종 수집에 사용한 명령어와 일치하게 유지하세요.
RECORD_COMMAND = r'''lerobot-record --robot.type=so101_follower --robot.port=COM6 --robot.id=my_follower --robot.cameras="{ cam0: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 15, fourcc: MJPG, backend: 700}, cam2: {type: opencv, index_or_path: 2, width: 640, height: 480, fps: 15, fourcc: MJPG, backend: 700}}" --teleop.type=so101_leader --teleop.port=COM7 --teleop.id=my_leader --display_data=false --dataset.repo_id=rimur01/so101-lab5-2cam --dataset.num_episodes=50 --dataset.episode_time_s=3600 --dataset.reset_time_s=5 --dataset.single_task="Pick up an object and place it in a target area" --dataset.private=false --dataset.push_to_hub=true'''
print(RECORD_COMMAND)


**녹화 팁:** 명령 실행 후 `cam0`과 `cam2`가 모두 연결되는지 확인합니다. 기록 인터페이스에서 성공한 시연을 마친 뒤 `n` 또는 오른쪽 화살표로 다음 에피소드로 넘어가고, 실패하거나 잘린 시연은 `r` 또는 왼쪽 화살표로 다시 기록하세요. 설치된 LeRobot 버전에서 표시되는 안내 키가 다르면 터미널 메시지를 우선하세요. 총 50개의 성공 에피소드가 저장되고 Hub 업로드가 완료되어야 합니다.


## 2. 데이터셋 메타데이터 확인

LeRobot v3에서는 에피소드 정보가 `meta/episodes/` 아래 Parquet 파일에 저장될 수 있으므로 v3 형식과 구형 JSONL 형식을 모두 확인합니다.


In [ ]:
def load_json(repo_id, filename, repo_type="dataset"):
    try:
        path = hf_hub_download(repo_id, filename, repo_type=repo_type)
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"{filename} 불러오기 실패: {type(e).__name__}: {e}")
        return None

info = load_json(DATASET, "meta/info.json")
if info:
    n_ep = info.get("total_episodes")
    fps = info.get("fps")
    cams = [k for k in info.get("features", {}) if k.startswith("observation.images")]
    print("데이터셋 포맷 버전:", info.get("codebase_version"))
    print("에피소드 수       :", n_ep)
    print("총 프레임 수      :", info.get("total_frames"))
    print("FPS               :", fps)
    print("카메라 features   :", cams)
    print("state 차원        :", info.get("features", {}).get("observation.state", {}).get("shape"))
    print("action 차원       :", info.get("features", {}).get("action", {}).get("shape"))
    print("에피소드 수 확인  :", "통과" if n_ep == 50 else f"미흡 ({n_ep}개; 50개 필요)")
    print("카메라 수 확인    :", "통과" if len(cams) >= 2 else f"미흡 ({len(cams)}개; 2개 필요)")
else:
    print("메타데이터를 읽지 못했습니다. DATASET repo_id와 업로드 상태를 확인하세요.")


## 2-1. 에피소드별 길이 및 태스크 확인

In [ ]:
ep_df = None
task_set = set()
lengths = None

try:
    pq_files = sorted(
        f for f in repo_files
        if f.startswith("meta/episodes/") and f.endswith(".parquet")
    )
    if pq_files:
        dfs = []
        for f in pq_files:
            path = hf_hub_download(repo_id=DATASET, filename=f, repo_type="dataset")
            dfs.append(pq.read_table(path).to_pandas())
        ep_df = pd.concat(dfs, ignore_index=True)
    elif "meta/episodes.jsonl" in repo_files:
        path = hf_hub_download(repo_id=DATASET, filename="meta/episodes.jsonl", repo_type="dataset")
        ep_df = pd.read_json(path, lines=True)
except Exception as e:
    print("에피소드 메타데이터를 불러오지 못했습니다:", type(e).__name__, e)

if ep_df is not None and {"episode_index", "length"}.issubset(ep_df.columns):
    ep_df = ep_df.sort_values("episode_index").reset_index(drop=True)
    lengths = ep_df[["episode_index", "length"]].copy()
    lengths["seconds"] = lengths["length"] / (fps or 30)
    median_len = lengths["length"].median()
    lengths["flag"] = lengths["length"].apply(
        lambda value: "짧음 의심" if value < 0.5 * median_len
        else ("김 의심" if value > 2.0 * median_len else "정상 범위")
    )
    print(
        f"에피소드 길이: 평균 {lengths['seconds'].mean():.1f}s, "
        f"최소 {lengths['seconds'].min():.1f}s, 최대 {lengths['seconds'].max():.1f}s"
    )
    print("에피소드 메타데이터 행 수:", len(ep_df))
    consecutive = ep_df["episode_index"].tolist() == list(range(len(ep_df)))
    print("인덱스 연속성 확인:", "통과" if consecutive else "주의: 인덱스 확인 필요")

    if "tasks" in ep_df.columns:
        for item in ep_df["tasks"]:
            if isinstance(item, str):
                task_set.add(item)
            elif item is not None:
                try:
                    task_set.update(str(x) for x in item)
                except TypeError:
                    task_set.add(str(item))
        print("기록된 태스크 문장:", task_set)
        print("TASK와 일치:", "통과" if task_set == {TASK} else "미흡 — TASK와 실제 기록 문장을 확인")
        print("영어 문장:", "통과" if all(t.isascii() for t in task_set) else "미흡 — 영어 태스크 문장을 사용")

    suspicious = lengths[lengths["flag"] != "정상 범위"]
    print("길이 이상 의심 에피소드:", suspicious["episode_index"].tolist() or "없음")
else:
    print("에피소드 길이 메타데이터를 확인하지 못했습니다. Hub 데이터셋을 확인하세요.")


## 2-2. 에피소드 길이 그래프

길이가 매우 짧거나 긴 에피소드는 잘림, 중단, 불필요하게 긴 정지일 수 있으므로 웹 뷰어에서 우선 확인합니다. 그래프만으로 성공 여부를 판정하지 마세요.


In [ ]:
if lengths is not None and not lengths.empty:
    color_map = {"짧음 의심": "tab:red", "김 의심": "tab:orange", "정상 범위": "tab:blue"}
    plt.figure(figsize=(12, 3.5))
    plt.bar(lengths["episode_index"], lengths["seconds"], color=lengths["flag"].map(color_map))
    plt.axhline(lengths["seconds"].median(), linestyle="--", color="gray", label="median")
    plt.xlabel("Episode index")
    plt.ylabel("Seconds")
    plt.title("Episode duration review")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("길이 이상 의심 에피소드:", lengths.loc[lengths["flag"] != "정상 범위", "episode_index"].tolist() or "없음")
else:
    print("길이 그래프를 그릴 데이터가 없습니다.")


## 3. 에피소드 검수

[Hugging Face LeRobot 데이터셋 뷰어](https://huggingface.co/spaces/lerobot/visualize_dataset)에 repo_id를 입력하고 50개 에피소드를 **모두 재생**하여 확인하세요.

각 에피소드에서 확인할 항목:

1. 태스크 성공 조건에 맞게 물체를 집어 목표 위치에 놓았는가?
2. 시작 동작 전부터 완료 후까지 기록되어 중간이 잘리지 않았는가?
3. `cam0`과 `cam2` 영상이 모두 정상인가? (검은 화면·멈춤·끊김 여부)
4. state/action이 실제 로봇 동작에 맞게 변하는가?

`O`는 직접 재생해 이상이 없음을 확인한 경우에만 기록합니다. 문제가 있으면 `X`와 메모를 남기고 다시 녹화한 뒤 데이터셋을 갱신하고 전부 재검수하세요.


In [ ]:
# [작성] 50개 에피소드를 실제로 재생한 뒤 ?를 O 또는 X로 바꾸고 필요한 메모를 남깁니다.
# 아직 재생 검수를 하지 않았다면 O로 미리 바꾸지 마세요.
REVIEW = """
0 | ? | 검수 전
1 | ? | 검수 전
2 | ? | 검수 전
3 | ? | 검수 전
4 | ? | 검수 전
5 | ? | 검수 전
6 | ? | 검수 전
7 | ? | 검수 전
8 | ? | 검수 전
9 | ? | 검수 전
10 | ? | 검수 전
11 | ? | 검수 전
12 | ? | 검수 전
13 | ? | 검수 전
14 | ? | 검수 전
15 | ? | 검수 전
16 | ? | 검수 전
17 | ? | 검수 전
18 | ? | 검수 전
19 | ? | 검수 전
20 | ? | 검수 전
21 | ? | 검수 전
22 | ? | 검수 전
23 | ? | 검수 전
24 | ? | 검수 전
25 | ? | 검수 전
26 | ? | 검수 전
27 | ? | 검수 전
28 | ? | 검수 전
29 | ? | 검수 전
30 | ? | 검수 전
31 | ? | 검수 전
32 | ? | 검수 전
33 | ? | 검수 전
34 | ? | 검수 전
35 | ? | 검수 전
36 | ? | 검수 전
37 | ? | 검수 전
38 | ? | 검수 전
39 | ? | 검수 전
40 | ? | 검수 전
41 | ? | 검수 전
42 | ? | 검수 전
43 | ? | 검수 전
44 | ? | 검수 전
45 | ? | 검수 전
46 | ? | 검수 전
47 | ? | 검수 전
48 | ? | 검수 전
49 | ? | 검수 전
"""

review_rows = []
for line in REVIEW.strip().splitlines():
    parts = [part.strip() for part in line.split("|")]
    if len(parts) >= 2:
        review_rows.append({
            "episode": int(parts[0]),
            "result": parts[1].upper(),
            "note": parts[2] if len(parts) > 2 else ""
        })
review_df = pd.DataFrame(review_rows)
print("검수 결과 집계:")
print(review_df["result"].value_counts(dropna=False).to_string())
print("아직 미검수/잘못된 결과가 있는 행:")
display(review_df[~review_df["result"].isin(["O", "X"])])
print("문제 있음(X) 에피소드:")
display(review_df[review_df["result"] == "X"])


## 3-2. 재녹화 기록

[작성] 실제 수집·검수 과정에서 다시 녹화한 에피소드 번호와 이유를 기록하세요. 재녹화가 없을 때만 `없음`이라고 작성합니다.


In [ ]:
RERECORD_NOTES = "[작성] 재녹화한 에피소드 번호와 이유를 기록하세요. 없으면 없음이라고 적으세요."
print("재녹화 기록:", RERECORD_NOTES)


## 4. 소감 및 과제 2 준비

다음은 예시 초안입니다. 실제 수집하면서 경험한 내용에 맞게 고친 뒤 제출하세요.


In [ ]:
# [작성] 본인의 실제 경험에 맞게 문장을 수정하세요.
REFLECTION_4_1 = (
    "물체의 시작 위치와 목표 위치를 일정하게 유지하면서도 시연마다 작은 위치 차이를 포함하려고 했다. "
    "각 시연에서 물체를 집고 이동한 뒤 놓는 동작 순서를 일관되게 유지했으며, 실패하거나 중간에 잘린 시연은 다시 기록했다."
)
REFLECTION_4_2 = (
    "데이터에 충분히 포함되지 않은 시작 위치, 물체 방향, 목표 위치에서는 ACT 정책의 성공률이 낮아질 수 있다. "
    "또한 카메라 시야가 달라지거나 물체가 가려지는 상황은 시연 데이터에서 다양하게 다루지 않았다면 어려울 수 있다."
)
print("4-1. 시연 일관성 유지 방법\n", REFLECTION_4_1)
print("\n4-2. ACT 학습 시 예상 어려움\n", REFLECTION_4_2)


## 5. 제출 전 자동 체크리스트

이 셀은 metadata와 검수표를 기반으로 형식적인 조건을 확인합니다. 자동 체크가 통과하더라도 모든 동영상을 직접 재생해 검수해야 합니다.


In [ ]:
review_complete = (
    review_df is not None
    and len(review_df) == 50
    and sorted(review_df["episode"].tolist()) == list(range(50))
    and review_df["result"].isin(["O", "X"]).all()
)
review_all_o = review_complete and (review_df["result"] == "O").all()
info_ok = bool(info) and info.get("total_episodes") == 50
camera_keys = [k for k in info.get("features", {}) if k.startswith("observation.images")] if info else []
cameras_ok = len(camera_keys) >= 2
public_ok = bool(is_public)
task_ok = bool(info) and task_set == {TASK} and all(t.isascii() for t in task_set)

check_items = {
    "Hub 저장소 접근 및 public 확인": public_ok,
    "에피소드 50개": info_ok,
    "카메라 feature 2개 이상": cameras_ok,
    "검수표에 에피소드 0~49 모두 기재": review_complete and sorted(review_df["episode"].tolist()) == list(range(50)),
    "검수표 결과가 모두 O": review_all_o,
    "TASK와 기록된 태스크 문장이 일치": task_ok,
}
for label, passed in check_items.items():
    print(("[통과] " if passed else "[미흡] ") + label)
print("\n모두 통과" if all(check_items.values()) else "\n미흡 항목을 해결하고, 전체 셀을 다시 실행한 뒤 커밋하세요.")


## GitHub 제출

1. 이 파일 이름을 `assignment_1.ipynb`로 유지합니다.
2. Jupyter에서 위에서부터 모든 셀을 실행하고 출력과 그래프가 저장되었는지 확인합니다.
3. 학번·이름, 최종 repo_id, 태스크 문장, 재녹화 기록, 소감과 50개 검수 결과를 실제 내용으로 완성합니다.
4. `assignments/assignment_1.ipynb` 경로로 GitHub 저장소에 커밋합니다.
5. 로봇 데이터 파일이나 동영상은 GitHub에 올리지 않습니다. 데이터는 Hugging Face Hub에만 저장합니다.

**중요:** 현재까지 확인한 카메라 테스트 성공을 50개 에피소드 완료로 간주하지 않습니다. 최종 Hub 데이터셋이 공개 상태이며 50개 에피소드와 두 카메라를 포함하고, 모든 에피소드를 직접 검수한 뒤에야 제출 완료로 볼 수 있습니다.
